# Importar ficheiros CRONO (cargas) para SQLite

Notebook unico com todo o processo:

1. Preparar a base de dados (tabelas `cargas_2025` / `cargas_2026` + indice unico anti-duplicados)
2. Ler os ficheiros CRONO da pasta `PASTA_FICHEIROS`
3. Inserir os dados, sem duplicar linhas - **lê o ano dos dados e insere na BD correta**
4. Validar - comparar ficheiros vs base de dados, por mes

In [ ]:
import os
import csv
import glob
import platform
import sqlite3
from datetime import datetime
from pathlib import Path

import pandas as pd

if platform.system() == "Windows":
    DB_PATH = r"C:\Users\LISARR\Documents\python\00.DB\2026.db"
    PASTA_FICHEIROS = Path(r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27")
elif platform.system() == "Darwin":
    DB_PATH = "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00.DB/2026.db"
    PASTA_FICHEIROS = Path("/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/2026_dados")

print("DB_PATH:", DB_PATH)
print("PASTA_FICHEIROS:", PASTA_FICHEIROS)

In [ ]:
COLUNAS_CARGAS = {
    "TIPO_SUMINISTRO": "Tipo Suministro",
    "BASE": "Base",
    "LANZADERA": "Lanzadera",
    "PUNTO_SUMINISTRO": "Punto Suministro",
    "RUTA": "Ruta",
    "PALES_TMS": "Palés TMS",
    "TEMPERATURA_REQUERIDA": "Temperatura",
    "AGENCIA": "Agencia",
    "TRANSPORTISTA": "Transportista",
    "DNI": "DNI",
    "PRECINTO": "Precinto",
    "TELEFONO": "Teléfono",
    "TRACTORA": "Tractora",
    "REMOLQUE": "Remolque",
    "FECHA_PREVISTA_POSICIONAMIENTO": "Fecha Prevista Posicionamiento",
    "HORA_PREVISTA_POSICIONAMIENTO": "Hora Prevista Posicionamiento",
    "FECHA_REAL_POSICIONAMIENTO": "Fecha Real Posicionamiento",
    "HORA_REAL_POSICIONAMIENTO": "Hora Real Posicionamiento",
    "FECHA_REAL_ENTRADA": "Fecha Real Entrada",
    "HORA_REAL_ENTRADA": "Hora Real Entrada",
    "MUELLE": "Muelle",
    "FECHA_PREVISTA_SALIDA": "Fecha Prevista Salida",
    "HORA_PREVISTA_SALIDA": "Hora Prevista Salida",
    "FECHA_REAL_SALIDA": "Fecha Real Salida",
    "HORA_REAL_SALIDA": "Hora Real Salida",
    "FECHA_PREVISTA_ENTREGA": "Fecha Prevista Entrega",
    "HORA_PREVISTA_ENTREGA": "Hora Prevista Entrega",
    "FECHA_REAL_ENTREGA": "Fecha Real Entrega",
    "HORA_REAL_ENTREGA": "Hora Real Entrega",
    "HORA_SALIDA_ENTREGA": "Hora Salida Entrega",
    "OBSERVACIONES": "Observaciones",
    "COMENTARIOS": "Comentarios",
    "ZONA": "Zona",
    "CLIENTE": "Cliente",
    "AUTORIZADO_AUTOCARGA": "Autorizado autocarga",
    "AUTOCARGA": "Autocarga",
    "HUECOS_TMS": "Huecos TMS",
    "HUECOS_CARGA": "Huecos carga",
    "HUECOS_DESCARGA": "Huecos descarga",
    "TEMPERATURA_MEDIDA1": "Temperatura",
    "TEMPERATURA_MEDIDA2": "Temperatura2",
    "MOTIVO": "Motivo",
    "ESTADO": "Estado",
    "ESTADO_MERCANCIA": "Estado mercancía",
    "ESTADO_CAJA": "Estado caja",
    "ESTADO_OLORES": "Estado olores",
    "ESTADO_LIMPIEZA_VEHICULO": "Estado limpieza vehículo",
    "ESTADO_VEHICULO_SECO": "Estado vehículo seco",
    "ESTADO_LIBRE_PLAGAS": "Estado libre de plagas",
}

ORDEM_CABECALHO_CARGAS = list(COLUNAS_CARGAS.values())
COLUNA_ORIGEM = "ficheiro_origem"
ANOS = ("2025", "2026")
CHAVE_UNICA_CARGAS = tuple(COLUNAS_CARGAS.keys())

In [ ]:
def preparar_base_dados(con):
    """Cria tabelas cargas_<ano> com indice unico."""
    colunas_sql = ",\n        ".join(f'"{c}" TEXT' for c in COLUNAS_CARGAS)
    chave_sql = ", ".join(f'COALESCE("{c}", \'\')' for c in CHAVE_UNICA_CARGAS)

    cur = con.cursor()
    for ano in ANOS:
        cur.execute(f'''
            CREATE TABLE IF NOT EXISTS cargas_{ano} (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                {colunas_sql},
                "{COLUNA_ORIGEM}" TEXT
            )
        ''')
        cur.execute(f'DROP INDEX IF EXISTS idx_cargas_{ano}_chave')
        cur.execute(
            f'CREATE UNIQUE INDEX idx_cargas_{ano}_chave '
            f'ON cargas_{ano}({chave_sql})'
        )
    con.commit()

pasta_db = os.path.dirname(DB_PATH)
if pasta_db and not os.path.exists(pasta_db):
    os.makedirs(pasta_db, exist_ok=True)

con = sqlite3.connect(DB_PATH)
preparar_base_dados(con)
con.close()
print("Base de dados, tabelas cargas e indice unico prontos.")

In [ ]:
def listar_ficheiros_excel(pasta):
    """Procura ficheiros .xls (tambem em subpastas)."""
    encontrados = glob.glob(os.path.join(pasta, "**", "*.xls"), recursive=True)
    vistos = set()
    ficheiros = []
    for caminho in encontrados:
        chave = os.path.normcase(os.path.abspath(caminho))
        if chave not in vistos:
            vistos.add(chave)
            ficheiros.append(caminho)
    return sorted(ficheiros)

def validar_data(valor):
    """Valida uma data DD/MM/AAAA e devolve o objeto datetime."""
    if not valor:
        return None

    texto = str(valor).strip()
    if not texto or texto.upper() == "N/D":
        return None

    try:
        return datetime.strptime(texto, "%d/%m/%Y")
    except ValueError:
        return None

def ler_linhas_csv_cargas(caminho_ficheiro):
    """Le um ficheiro CRONO (tab-separated, ISO-8859-1)."""
    with open(caminho_ficheiro, encoding="iso-8859-1", newline="") as f:
        linhas_ficheiro = list(csv.reader(f, delimiter="\t"))

    if not linhas_ficheiro:
        return [], []

    cabecalho = linhas_ficheiro[0]
    linhas = linhas_ficheiro[1:]

    n_colunas = len(cabecalho)
    linhas_normalizadas = []
    for valores in linhas:
        if len(valores) < n_colunas:
            valores = valores + [None] * (n_colunas - len(valores))
        elif len(valores) > n_colunas:
            valores = valores[:n_colunas]
        linhas_normalizadas.append(valores)

    return cabecalho, linhas_normalizadas

ficheiros = listar_ficheiros_excel(PASTA_FICHEIROS)
print(f"Ficheiros encontrados: {len(ficheiros)}")

In [ ]:
# === NOVAS FUNÇÕES: extrair ano e montar linha ===

def extrair_ano_e_data_cargas(valores):
    """
    Extrai ano e data de uma linha de cargas.
    Procura a primeira coluna de data válida (FECHA_PREVISTA_POSICIONAMIENTO).
    Devolve (ano_str, data_str) ou (None, None) se não encontrar.
    """
    # Índice de FECHA_PREVISTA_POSICIONAMIENTO no array de valores
    colunas_ordem = list(COLUNAS_CARGAS.keys())
    idx_fecha = colunas_ordem.index("FECHA_PREVISTA_POSICIONAMIENTO")
    
    if idx_fecha < len(valores):
        data_str = valores[idx_fecha]
        data_obj = validar_data(data_str)
        if data_obj:
            ano_str = str(data_obj.year)
            return ano_str, data_str
    
    return None, None


def montar_linha_cargas(valores, nome_ficheiro):
    """
    Monta uma linha pronta para INSERT na BD.
    Devolve tupla (valor1, valor2, ..., ficheiro_origem).
    """
    colunas_ordem = list(COLUNAS_CARGAS.keys())
    linha = tuple(valores[:len(colunas_ordem)]) + (nome_ficheiro,)
    return linha


# Preparar statements SQL de INSERT para cada ano
sql_insercao = {}
for ano in ANOS:
    colunas = ",".join(f'"{c}"' for c in COLUNAS_CARGAS.keys())
    placeholders = ",".join(["?"] * (len(COLUNAS_CARGAS) + 1))  # +1 para ficheiro_origem
    sql_insercao[ano] = f'INSERT INTO cargas_{ano} ({colunas}, "{COLUNA_ORIGEM}") VALUES ({placeholders})'

print("Funções e statements SQL preparados.")

In [ ]:
# ==========================
# IMPORTAR COM VERIFICAÇÃO DE BD
# ==========================

contagem_novas = {ano: 0 for ano in ANOS}
total_duplicadas = 0
total_sem_data = 0
total_sem_bd = 0
total_processados = 0
total_cabecalho_invalido = 0
total_erros = 0

linhas_sem_bd = []  # Guardar linhas de anos sem BD

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

# Verificar quais as BDs existentes
cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name LIKE 'cargas_%'")
tabelas_existentes = {row[0] for row in cur.fetchall()}
anos_disponiveis = {int(t.split('_')[1]) for t in tabelas_existentes if t.startswith('cargas_')}
anos_disponiveis_str = {str(a) for a in anos_disponiveis}

# Processar ficheiros
for i, caminho in enumerate(ficheiros, start=1):
    nome_ficheiro = os.path.basename(caminho)

    try:
        cabecalho, linhas = ler_linhas_csv_cargas(caminho)

        if cabecalho != ORDEM_CABECALHO_CARGAS:
            total_cabecalho_invalido += 1
            print(
                f"[{i}/{len(ficheiros)}] [AVISO] "
                f"cabecalho de '{nome_ficheiro}' diferente do esperado - ficheiro ignorado."
            )
            continue

        novas_ficheiro = {ano: 0 for ano in ANOS}
        duplicadas_ficheiro = 0
        sem_data_ficheiro = 0
        sem_bd_ficheiro = 0

        for valores in linhas:
            if all(v is None or str(v).strip() == "" for v in valores):
                continue

            ano, _ = extrair_ano_e_data_cargas(valores)

            if ano not in ANOS:
                sem_data_ficheiro += 1
                continue

            # Verificar se a BD para este ano existe
            if ano not in anos_disponiveis_str:
                sem_bd_ficheiro += 1
                total_sem_bd += 1
                linha = montar_linha_cargas(valores, nome_ficheiro)
                linhas_sem_bd.append((ano, linha))
                continue

            # Inserir normalmente
            linha = montar_linha_cargas(valores, nome_ficheiro)
            cur.execute(sql_insercao[ano], linha)

            if cur.rowcount == 1:
                novas_ficheiro[ano] += 1
            else:
                duplicadas_ficheiro += 1

        # Confirma apenas este ficheiro
        con.commit()
        total_processados += 1

        for ano in ANOS:
            contagem_novas[ano] += novas_ficheiro[ano]

        total_duplicadas += duplicadas_ficheiro
        total_sem_data += sem_data_ficheiro

        agora = datetime.now().isoformat(timespec="seconds")
        resumo = f"[{i}/{len(ficheiros)}] {nome_ficheiro} -> "
        partes = []
        if novas_ficheiro['2025'] > 0:
            partes.append(f"{novas_ficheiro['2025']} novas 2025")
        if novas_ficheiro['2026'] > 0:
            partes.append(f"{novas_ficheiro['2026']} novas 2026")
        if duplicadas_ficheiro > 0:
            partes.append(f"{duplicadas_ficheiro} duplicadas")
        if sem_bd_ficheiro > 0:
            partes.append(f"[SEM BD] {sem_bd_ficheiro}")
        if sem_data_ficheiro > 0:
            partes.append(f"{sem_data_ficheiro} sem data")
        
        print(resumo + " | ".join(partes) if partes else resumo + "nenhuma linha processada")

    except Exception as erro:
        con.rollback()
        total_erros += 1
        print(f"[{i}/{len(ficheiros)}] [ERRO] '{nome_ficheiro}': {erro}")

con.close()

print("\n" + "="*70)
print("RESUMO FINAL")
print("="*70)
print(f"Ficheiros encontrados:                  {len(ficheiros)}")
print(f"Ficheiros processados:                  {total_processados}")
print(f"Ficheiros com erro:                     {total_erros}")
print(f"Linhas novas inseridas em 2025:         {contagem_novas['2025']}")
print(f"Linhas novas inseridas em 2026:         {contagem_novas['2026']}")
print(f"Linhas ja existentes:                   {total_duplicadas}")
print(f"Linhas SEM base de dados:               {total_sem_bd}")
print(f"Linhas sem data valida:                 {total_sem_data}")
print("="*70)

In [ ]:
# === MOSTRAR LINHAS SEM BD ===

if linhas_sem_bd:
    print(f"\n{'='*70}")
    print(f"LINHAS SEM BASE DE DADOS ({len(linhas_sem_bd)} total):")
    print(f"{'='*70}\n")
    
    colunas_ordem = list(COLUNAS_CARGAS.keys())
    dados = []
    for ano, linha in linhas_sem_bd:
        dados.append({
            'ANO_FALTA': ano,
            **{col: linha[i] if i < len(linha) else None 
               for i, col in enumerate(colunas_ordem)}
        })
    
    df_sem_bd = pd.DataFrame(dados)
    
    for ano in sorted(df_sem_bd['ANO_FALTA'].unique()):
        print(f"\nAno {ano} ({len(df_sem_bd[df_sem_bd['ANO_FALTA'] == ano])} linhas):")
        df_ano = df_sem_bd[df_sem_bd['ANO_FALTA'] == ano].drop('ANO_FALTA', axis=1)
        df_ano.head(100)
else:
    print("\nTodas as linhas foram processadas com sucesso (nenhuma sem BD).")